In [1]:
#!kaggle competitions download -c springleaf-marketing-response

In [2]:
#!dir

In [3]:
#import zipfile

#with zipfile.ZipFile('springleaf-marketing-response.zip', 'r') as zip_ref:
#    zip_ref.extractall('springleaf_data')


In [4]:
#!dir

In [5]:
#!head -50000 train/train.csv > smalltrain.csv
#!head -50000 test/test.csv > smalltest.csv

# REDUCING THE COLUMNS

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [7]:
traindf = pd.read_csv("smalltrain.csv", low_memory=False)
testdf = pd.read_csv("smalltest.csv", low_memory=False)
print(f"Starting shape: {traindf.shape}")

Starting shape: (49999, 1934)


In [8]:
copydf = traindf.copy()
print("done")
#just a backup dataframe to compare to if needed
#traindf=copydf.copy

done


In [9]:
suspect_cols = [8,9,10,11,12,43,157,196,214,225,228,229,231,235,238]
# when running code to read csv it stated these columns contained mixed datatypes
col_names = traindf.columns[suspect_cols]

for col in col_names:
    print(f"--- {col} ---")
    print(traindf[col].apply(type).value_counts(), "\n")

    print(f"Unique values in '{col}':")
    print(traindf[col].unique())
    print("-" * 40)


--- VAR_0008 ---
VAR_0008
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0008':
[False nan]
----------------------------------------
--- VAR_0009 ---
VAR_0009
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0009':
[False nan]
----------------------------------------
--- VAR_0010 ---
VAR_0010
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0010':
[False nan]
----------------------------------------
--- VAR_0011 ---
VAR_0011
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0011':
[False nan]
----------------------------------------
--- VAR_0012 ---
VAR_0012
<class 'bool'>     49980
<class 'float'>       19
Name: count, dtype: int64 

Unique values in 'VAR_0012':
[False nan]
----------------------------------------
--- VAR_0043 ---
VAR_0043
<class 'bool'>     49980
<class 'float'>    

In [10]:
# Get non-numeric (categorical) columns
non_numeric_cols = traindf.select_dtypes(exclude=['number']).columns.tolist()

# Count of categorical columns
categorical_count = len(non_numeric_cols)

print(f"Non-numeric (categorical) columns ({categorical_count} total):")
print(non_numeric_cols)

Non-numeric (categorical) columns (51 total):
['VAR_0001', 'VAR_0005', 'VAR_0008', 'VAR_0009', 'VAR_0010', 'VAR_0011', 'VAR_0012', 'VAR_0043', 'VAR_0044', 'VAR_0073', 'VAR_0075', 'VAR_0156', 'VAR_0157', 'VAR_0158', 'VAR_0159', 'VAR_0166', 'VAR_0167', 'VAR_0168', 'VAR_0169', 'VAR_0176', 'VAR_0177', 'VAR_0178', 'VAR_0179', 'VAR_0196', 'VAR_0200', 'VAR_0202', 'VAR_0204', 'VAR_0214', 'VAR_0216', 'VAR_0217', 'VAR_0222', 'VAR_0226', 'VAR_0229', 'VAR_0230', 'VAR_0232', 'VAR_0236', 'VAR_0237', 'VAR_0239', 'VAR_0274', 'VAR_0283', 'VAR_0305', 'VAR_0325', 'VAR_0342', 'VAR_0352', 'VAR_0353', 'VAR_0354', 'VAR_0404', 'VAR_0466', 'VAR_0467', 'VAR_0493', 'VAR_1934']


In [11]:
# above columns seem to be primarily boolean, let's see which are likely all boolean
# Coerce columns with mostly boolean-like data and then we'll look at the two remaining suspicious columns
for col in col_names:
    if col not in ["VAR_0157", "VAR_0214"]:
        traindf[col] = traindf[col].astype(str).map(lambda x: x.strip() in ['1', 'True', 'Y', 'Yes'])


In [12]:
boolean_like = {True, False, 1, 0, '1', '0', 'True', 'False', 'Y', 'N', 'Yes', 'No'}
# Loop through suspect columns and print unusual values
for col in col_names:
    unique_vals = set(traindf[col].dropna().unique())
    unusual_vals = unique_vals - boolean_like
    if unusual_vals:
        print(f"Column {col} has unusual values: {unusual_vals}\n")

Column VAR_0157 has unusual values: {'11FEB12:00:00:00', '25OCT11:00:00:00', '29MAY11:00:00:00', '01MAY12:00:00:00', '07OCT11:00:00:00', '16JAN09:00:00:00', '09JUN12:00:00:00', '18SEP12:00:00:00', '16JUN12:00:00:00', '04JUL12:00:00:00', '24MAY12:00:00:00', '31JUL12:00:00:00', '09JAN11:00:00:00', '02SEP11:00:00:00', '12OCT11:00:00:00', '05AUG12:00:00:00', '06OCT11:00:00:00', '12JAN12:00:00:00', '09APR12:00:00:00', '10JUL12:00:00:00', '12NOV11:00:00:00', '30AUG12:00:00:00', '19MAY12:00:00:00', '09AUG12:00:00:00', '17DEC11:00:00:00', '05JUN10:00:00:00', '28JAN12:00:00:00', '14FEB12:00:00:00', '20JUL12:00:00:00', '30JUN12:00:00:00', '02DEC11:00:00:00', '23JUL12:00:00:00', '30JUL11:00:00:00', '30NOV11:00:00:00', '18DEC11:00:00:00', '10APR12:00:00:00', '19AUG12:00:00:00', '16DEC11:00:00:00', '25MAR12:00:00:00', '13JUL12:00:00:00', '01JUL12:00:00:00', '07FEB12:00:00:00', '13JAN12:00:00:00', '06AUG12:00:00:00', '09MAR12:00:00:00', '20AUG12:00:00:00', '30JAN10:00:00:00', '29NOV11:00:00:00', '10

In [13]:
for col in col_names:
    print(f"--- {col} ---")
    print(traindf[col].apply(type).value_counts(), "\n")

    print(f"Unique values in '{col}':")
    print(traindf[col].unique())
    print("-" * 40)

--- VAR_0008 ---
VAR_0008
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0008':
[False]
----------------------------------------
--- VAR_0009 ---
VAR_0009
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0009':
[False]
----------------------------------------
--- VAR_0010 ---
VAR_0010
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0010':
[False]
----------------------------------------
--- VAR_0011 ---
VAR_0011
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0011':
[False]
----------------------------------------
--- VAR_0012 ---
VAR_0012
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0012':
[False]
----------------------------------------
--- VAR_0043 ---
VAR_0043
<class 'bool'>    49999
Name: count, dtype: int64 

Unique values in 'VAR_0043':
[False]
----------------------------------------
--- VAR_0157 ---
VAR_0157
<class 'float'>    49673
<class 'str'>

In [14]:
missing_pct = traindf.isnull().mean()
high_missing = missing_pct[missing_pct > 0.60].index
traindf.drop(columns=high_missing, inplace=True)
print(f"Dropped {len(high_missing)} high-missing columns\n{high_missing}")

Dropped 24 high-missing columns
Index(['VAR_0073', 'VAR_0074', 'VAR_0156', 'VAR_0157', 'VAR_0158', 'VAR_0159',
       'VAR_0166', 'VAR_0167', 'VAR_0168', 'VAR_0169', 'VAR_0176', 'VAR_0177',
       'VAR_0178', 'VAR_0179', 'VAR_0205', 'VAR_0206', 'VAR_0207', 'VAR_0208',
       'VAR_0209', 'VAR_0210', 'VAR_0211', 'VAR_0213', 'VAR_0214', 'VAR_0840'],
      dtype='object')


In [15]:
#looks like those two problematic columns had less than 10% data filled anyway, as they've been dropped we move on
nunique = traindf.nunique()
constant_cols = nunique[nunique == 1].index
traindf.drop(columns=constant_cols, inplace=True)
print(f"Dropped {len(constant_cols)} columns which only have one value")

Dropped 54 columns which only have one value


In [16]:
# Gets a list of duplicate columns
def get_duplicate_columns(df):
    duplicates = set()
    cols = df.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if cols[j] not in duplicates and df[cols[i]].equals(df[cols[j]]):
                duplicates.add(cols[j])
    return list(duplicates)

# Drops those columns
dupe_cols = get_duplicate_columns(traindf)
traindf.drop(columns=dupe_cols, inplace=True)

print(f"Dropped {len(dupe_cols)} duplicate columns: {dupe_cols}")

Dropped 11 duplicate columns: ['VAR_0013', 'VAR_0238', 'VAR_0357', 'VAR_1036', 'VAR_0398', 'VAR_0130', 'VAR_0228', 'VAR_0182', 'VAR_0201', 'VAR_0181', 'VAR_0512']


In [17]:
# Get non-numeric (categorical) columns
non_numeric_cols = traindf.select_dtypes(exclude=['number']).columns.tolist()

# Count of categorical columns
categorical_count = len(non_numeric_cols)

print(f"Non-numeric (categorical) columns ({categorical_count} total):")
print(non_numeric_cols)

Non-numeric (categorical) columns (24 total):
['VAR_0001', 'VAR_0005', 'VAR_0075', 'VAR_0200', 'VAR_0204', 'VAR_0217', 'VAR_0226', 'VAR_0230', 'VAR_0232', 'VAR_0236', 'VAR_0237', 'VAR_0274', 'VAR_0283', 'VAR_0305', 'VAR_0325', 'VAR_0342', 'VAR_0352', 'VAR_0353', 'VAR_0354', 'VAR_0404', 'VAR_0466', 'VAR_0467', 'VAR_0493', 'VAR_1934']


In [18]:
for col in non_numeric_cols:
    print(f"\nColumn: {col}")
    print("-" * (len(col) + 9))
    print(traindf[col].value_counts(dropna=False).head(30))


Column: VAR_0001
-----------------
VAR_0001
R    29280
H    20532
Q      187
Name: count, dtype: int64

Column: VAR_0005
-----------------
VAR_0005
B    24747
C    18514
N     5727
S     1011
Name: count, dtype: int64

Column: VAR_0075
-----------------
VAR_0075
22SEP10:00:00:00    428
23SEP10:00:00:00    299
23NOV11:00:00:00    254
15NOV11:00:00:00    252
06DEC11:00:00:00    251
22NOV11:00:00:00    246
07DEC11:00:00:00    241
09SEP11:00:00:00    221
08DEC11:00:00:00    220
06OCT11:00:00:00    213
10NOV11:00:00:00    204
08NOV11:00:00:00    203
18OCT11:00:00:00    202
19OCT11:00:00:00    202
05OCT11:00:00:00    200
07OCT11:00:00:00    199
04OCT11:00:00:00    198
17SEP10:00:00:00    197
13DEC11:00:00:00    196
04NOV11:00:00:00    193
11OCT11:00:00:00    184
01NOV11:00:00:00    183
09NOV11:00:00:00    180
19JUN12:00:00:00    179
19JAN12:00:00:00    177
14DEC11:00:00:00    174
20OCT11:00:00:00    173
17MAY12:00:00:00    173
29NOV11:00:00:00    172
21OCT11:00:00:00    172
Name: count, dty

In [19]:
# Looks like multiple boolean columns are stored as categorical still, so let's address those.

suspect_cols2 = ["VAR_0226", "VAR_0230", "VAR_0232", "VAR_0236"]
print(traindf[suspect_cols2].dtypes)
for col in suspect_cols2:
    unique_vals = set(traindf[col].dropna().unique())
    unusual_vals = unique_vals - boolean_like
    if unusual_vals:
        print(f"Column {col} has unusual values: {unusual_vals}\n")


VAR_0226    bool
VAR_0230    bool
VAR_0232    bool
VAR_0236    bool
dtype: object


In [20]:
#Let's see how many columns have null values
null_summary = traindf.isnull().sum()
print(null_summary[null_summary > 0])

VAR_0006     19
VAR_0007     19
VAR_0014     19
VAR_0015     19
VAR_0016     19
           ... 
VAR_0522    311
VAR_0523    311
VAR_0524    311
VAR_0525    311
VAR_0531    311
Length: 435, dtype: int64


In [21]:
from sklearn.impute import SimpleImputer
#handle them replacing with median
numeric_cols = traindf.select_dtypes(include=[np.number]).columns
num_imputer = SimpleImputer(strategy="median")
traindf[numeric_cols] = num_imputer.fit_transform(traindf[numeric_cols])
print("Done")

Done


In [22]:
cat_imputer = SimpleImputer(strategy="most_frequent")
traindf[non_numeric_cols] = cat_imputer.fit_transform(traindf[non_numeric_cols])


In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
non_numeric_cols = traindf.select_dtypes(exclude=[np.number]).columns.tolist()

for col in non_numeric_cols:
    le = LabelEncoder()
    traindf[col] = le.fit_transform(traindf[col].astype(str))
    label_encoders[col] = le

In [24]:
print(f"Remaining shape: {traindf.shape}")

Remaining shape: (49999, 1845)


In [25]:
# There's so many columns, for now I'll only look at numeric columns for correlation to target
numeric_cols = traindf.select_dtypes(include=['number']).columns

correlations = traindf[numeric_cols].corrwith(traindf['target']).abs()

top_corr = correlations.sort_values(ascending=False)
print(top_corr.head(20))

target      1.000000
VAR_0105    0.209993
VAR_0505    0.207471
VAR_0121    0.205737
VAR_0104    0.203506
VAR_0120    0.200778
VAR_0886    0.200230
VAR_0145    0.196768
VAR_0113    0.196746
VAR_0103    0.193520
VAR_0119    0.193111
VAR_0015    0.192504
VAR_0144    0.192084
VAR_0017    0.191571
VAR_0137    0.189999
VAR_0232    0.188385
VAR_0006    0.188160
VAR_0795    0.186871
VAR_0503    0.185934
VAR_0143    0.185781
dtype: float64


In [26]:
#Now I look at the correlation to each other
numeric_df = traindf.select_dtypes(include=[np.number])

corr_matrix = numeric_df.corr().abs()  # Use absolute value to ignore sign

# Fill diagonal with NaN so self-correlation doesn't affect the average
np.fill_diagonal(corr_matrix.values, np.nan)

# Compute average correlation for each column
avg_corr = corr_matrix.mean()

most_unique_cols = avg_corr.sort_values().index[:50]
print("Top 50 most unique columns based on average correlation:")
print(most_unique_cols.tolist())


Top 50 most unique columns based on average correlation:
['VAR_0411', 'VAR_0463', 'VAR_0191', 'VAR_0449', 'VAR_0395', 'VAR_0340', 'VAR_0180', 'VAR_0459', 'ID', 'VAR_0445', 'VAR_0396', 'VAR_0192', 'VAR_1427', 'VAR_0521', 'VAR_0437', 'VAR_0386', 'VAR_0909', 'VAR_0310', 'VAR_0075', 'VAR_0397', 'VAR_0193', 'VAR_0226', 'VAR_0333', 'VAR_0270', 'VAR_0502', 'VAR_0330', 'VAR_0331', 'VAR_0303', 'VAR_0217', 'VAR_0243', 'VAR_0289', 'VAR_0264', 'VAR_0194', 'VAR_0230', 'VAR_0399', 'VAR_0436', 'VAR_0107', 'VAR_0200', 'VAR_0098', 'VAR_0195', 'VAR_0138', 'VAR_0443', 'VAR_0377', 'VAR_0551', 'VAR_0387', 'VAR_0524', 'VAR_0114', 'VAR_0393', 'VAR_0496', 'VAR_0371']


In [27]:
# Get top most unique columns (lowest average correlation to others)
top_unique_cols = avg_corr.sort_values().index[:600]

# Get top most target-correlated columns (highest abs correlation to target)
top_target_corr_cols = correlations.sort_values(ascending=False).index[:600]

# All so we can make this list! Played with index numbers till I got a good 
# number of columns to train off of.


best_of_both = list(set(top_unique_cols) & set(top_target_corr_cols))

# Also I noticed the ID column snuck in there so let's just make sure it doesn't stay
excluded_cols = {'ID', 'target'}

top_unique_cols = [col for col in top_unique_cols if col not in excluded_cols][:75]
top_target_corr_cols = [col for col in top_target_corr_cols if col not in excluded_cols][:75]
best_of_both = [col for col in best_of_both if col not in excluded_cols]

print(f"Found {len(best_of_both)} columns that are both unique and correlated with target:")
print(best_of_both)

Found 80 columns that are both unique and correlated with target:
['VAR_1564', 'VAR_0060', 'VAR_0128', 'VAR_0795', 'VAR_0077', 'VAR_0096', 'VAR_0127', 'VAR_0071', 'VAR_0173', 'VAR_1570', 'VAR_0079', 'VAR_0134', 'VAR_0095', 'VAR_1548', 'VAR_0101', 'VAR_0088', 'VAR_0097', 'VAR_0926', 'VAR_0078', 'VAR_0164', 'VAR_0053', 'VAR_1550', 'VAR_0089', 'VAR_0160', 'VAR_0080', 'VAR_1547', 'VAR_0117', 'VAR_0233', 'VAR_0082', 'VAR_0035', 'VAR_0052', 'VAR_0174', 'VAR_0154', 'VAR_0076', 'VAR_0129', 'VAR_0110', 'VAR_0050', 'VAR_0051', 'VAR_0187', 'VAR_0503', 'VAR_1571', 'VAR_0235', 'VAR_0133', 'VAR_0087', 'VAR_0081', 'VAR_0069', 'VAR_1934', 'VAR_0758', 'VAR_0170', 'VAR_0142', 'VAR_0059', 'VAR_0066', 'VAR_0511', 'VAR_0085', 'VAR_0083', 'VAR_0126', 'VAR_0150', 'VAR_0768', 'VAR_1575', 'VAR_0070', 'VAR_1549', 'VAR_0054', 'VAR_0068', 'VAR_0625', 'VAR_1576', 'VAR_0360', 'VAR_0111', 'VAR_0162', 'VAR_0109', 'VAR_0172', 'VAR_0084', 'VAR_0072', 'VAR_0231', 'VAR_0359', 'VAR_0086', 'VAR_0620', 'VAR_0141', 'VAR_0234

In [28]:
print (top_unique_cols)
print ()
print ()
print (top_target_corr_cols)

['VAR_0411', 'VAR_0463', 'VAR_0191', 'VAR_0449', 'VAR_0395', 'VAR_0340', 'VAR_0180', 'VAR_0459', 'VAR_0445', 'VAR_0396', 'VAR_0192', 'VAR_1427', 'VAR_0521', 'VAR_0437', 'VAR_0386', 'VAR_0909', 'VAR_0310', 'VAR_0075', 'VAR_0397', 'VAR_0193', 'VAR_0226', 'VAR_0333', 'VAR_0270', 'VAR_0502', 'VAR_0330', 'VAR_0331', 'VAR_0303', 'VAR_0217', 'VAR_0243', 'VAR_0289', 'VAR_0264', 'VAR_0194', 'VAR_0230', 'VAR_0399', 'VAR_0436', 'VAR_0107', 'VAR_0200', 'VAR_0098', 'VAR_0195', 'VAR_0138', 'VAR_0443', 'VAR_0377', 'VAR_0551', 'VAR_0387', 'VAR_0524', 'VAR_0114', 'VAR_0393', 'VAR_0496', 'VAR_0371', 'VAR_0315', 'VAR_0428', 'VAR_0388', 'VAR_0284', 'VAR_0392', 'VAR_0439', 'VAR_0295', 'VAR_0441', 'VAR_0447', 'VAR_0313', 'VAR_0519', 'VAR_0452', 'VAR_0293', 'VAR_0278', 'VAR_0389', 'VAR_0460', 'VAR_0442', 'VAR_0204', 'VAR_0280', 'VAR_1845', 'VAR_1847', 'VAR_1846', 'VAR_1850', 'VAR_0414', 'VAR_0183', 'VAR_1848']


['VAR_0105', 'VAR_0505', 'VAR_0121', 'VAR_0104', 'VAR_0120', 'VAR_0886', 'VAR_0145', 'VAR_0113', 

# MODELS

In [34]:

# Function to evaluate model on a subset of features
def evaluate_model(model, df, features, label='target', test_size=0.3, random_state=42):
    X = df[features]
    y = df[label]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)

    # Predict probabilities or decision scores for AUC-ROC
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_proba = model.decision_function(X_test)
    else:
        y_proba = None

    print(f"\n--- Evaluation for {model.__class__.__name__} using {len(features)} features ---")
    print(classification_report(y_test, y_pred, digits=4))
    
    if y_proba is not None:
        auc = roc_auc_score(y_test, y_proba)
        print(f"AUC-ROC: {auc:.4f}")
    else:
        print("AUC-ROC could not be computed (model lacks probability output).")

    return model


### Bestofboth

In [54]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
evaluate_model(rf, traindf, best_of_both)


--- Evaluation for RandomForestClassifier using 80 features ---
              precision    recall  f1-score   support

         0.0     0.7880    0.9430    0.8586     11517
         1.0     0.4610    0.1614    0.2390      3483

    accuracy                         0.7615     15000
   macro avg     0.6245    0.5522    0.5488     15000
weighted avg     0.7121    0.7615    0.7147     15000

AUC-ROC: 0.6716


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [55]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
evaluate_model(dt, traindf, best_of_both)



--- Evaluation for DecisionTreeClassifier using 80 features ---
              precision    recall  f1-score   support

         0.0     0.7949    0.7907    0.7928     11517
         1.0     0.3198    0.3253    0.3225      3483

    accuracy                         0.6827     15000
   macro avg     0.5573    0.5580    0.5577     15000
weighted avg     0.6846    0.6827    0.6836     15000

AUC-ROC: 0.5404


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', DecisionTreeClassifier(random_state=42))])

In [53]:
def evaluate_model(model, df, features, label='target', test_size=0.3, random_state=42):
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    X = df[features]
    y = df[label]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # Fit pipeline
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(f"\n--- Evaluation for {model.__class__.__name__} using {len(features)} features ---")
    print(classification_report(y_test, y_pred, digits=4))

    # Predict probabilities or decision scores for AUC-ROC
    y_proba = None
    try:
        y_proba = pipe.predict_proba(X_test)[:, 1]
    except AttributeError:
        try:
            y_proba = pipe.decision_function(X_test)
        except AttributeError:
            pass

    if y_proba is not None:
        auc = roc_auc_score(y_test, y_proba)
        print(f"AUC-ROC: {auc:.4f}")
    else:
        print("AUC-ROC could not be computed (model lacks probability or decision output).")

    return pipe

In [42]:
from xgboost import XGBClassifier
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
evaluate_model(xgb, traindf, best_of_both)



--- Evaluation for XGBClassifier using 80 features ---
              precision    recall  f1-score   support

         0.0     0.7908    0.9428    0.8601     11517
         1.0     0.4811    0.1754    0.2571      3483

    accuracy                         0.7646     15000
   macro avg     0.6360    0.5591    0.5586     15000
weighted avg     0.7189    0.7646    0.7201     15000

AUC-ROC: 0.6894


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [43]:
from sklearn.linear_model import LogisticRegression

evaluate_model(LogisticRegression(max_iter=1000), traindf, best_of_both)



--- Evaluation for LogisticRegression using 80 features ---
              precision    recall  f1-score   support

         0.0     0.7880    0.9667    0.8682     11517
         1.0     0.5591    0.1398    0.2237      3483

    accuracy                         0.7747     15000
   macro avg     0.6735    0.5532    0.5460     15000
weighted avg     0.7348    0.7747    0.7186     15000

AUC-ROC: 0.6999


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

### top unique

In [44]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
evaluate_model(rf, traindf, top_unique_cols)


--- Evaluation for RandomForestClassifier using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7693    0.9928    0.8669     11517
         1.0     0.3986    0.0158    0.0304      3483

    accuracy                         0.7659     15000
   macro avg     0.5839    0.5043    0.4486     15000
weighted avg     0.6832    0.7659    0.6727     15000

AUC-ROC: 0.5651


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [45]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
evaluate_model(dt, traindf, top_unique_cols)



--- Evaluation for DecisionTreeClassifier using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7719    0.7582    0.7650     11517
         1.0     0.2448    0.2593    0.2518      3483

    accuracy                         0.6423     15000
   macro avg     0.5084    0.5087    0.5084     15000
weighted avg     0.6495    0.6423    0.6458     15000

AUC-ROC: 0.5087


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', DecisionTreeClassifier(random_state=42))])

In [46]:
from xgboost import XGBClassifier
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
evaluate_model(xgb, traindf, top_unique_cols)


--- Evaluation for XGBClassifier using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7723    0.9807    0.8641     11517
         1.0     0.4080    0.0439    0.0793      3483

    accuracy                         0.7632     15000
   macro avg     0.5902    0.5123    0.4717     15000
weighted avg     0.6877    0.7632    0.6819     15000

AUC-ROC: 0.5911


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [47]:
from sklearn.linear_model import LogisticRegression

evaluate_model(LogisticRegression(max_iter=1000), traindf, top_unique_cols)



--- Evaluation for LogisticRegression using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7681    0.9992    0.8686     11517
         1.0     0.5000    0.0026    0.0051      3483

    accuracy                         0.7678     15000
   macro avg     0.6341    0.5009    0.4369     15000
weighted avg     0.7059    0.7678    0.6681     15000

AUC-ROC: 0.5420


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

Well that was awful

### top corr

In [48]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
evaluate_model(rf, traindf, top_target_corr_cols)


--- Evaluation for RandomForestClassifier using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7947    0.9522    0.8664     11517
         1.0     0.5417    0.1866    0.2776      3483

    accuracy                         0.7745     15000
   macro avg     0.6682    0.5694    0.5720     15000
weighted avg     0.7360    0.7745    0.7297     15000

AUC-ROC: 0.7188


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', RandomForestClassifier(n_jobs=-1, random_state=42))])

In [56]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
evaluate_model(dt, traindf, top_target_corr_cols)



--- Evaluation for DecisionTreeClassifier using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7991    0.7889    0.7940     11517
         1.0     0.3301    0.3440    0.3369      3483

    accuracy                         0.6856     15000
   macro avg     0.5646    0.5664    0.5654     15000
weighted avg     0.6902    0.6856    0.6878     15000

AUC-ROC: 0.5664


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', DecisionTreeClassifier(random_state=42))])

In [57]:
from xgboost import XGBClassifier
xgb = XGBClassifier(eval_metric='logloss', random_state=42)
evaluate_model(xgb, traindf, top_target_corr_cols)


--- Evaluation for XGBClassifier using 75 features ---
              precision    recall  f1-score   support

         0.0     0.8022    0.9318    0.8622     11517
         1.0     0.5160    0.2403    0.3279      3483

    accuracy                         0.7713     15000
   macro avg     0.6591    0.5861    0.5950     15000
weighted avg     0.7358    0.7713    0.7381     15000

AUC-ROC: 0.7119


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='logloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [58]:
from sklearn.linear_model import LogisticRegression

evaluate_model(LogisticRegression(max_iter=1000), traindf, top_target_corr_cols)



--- Evaluation for LogisticRegression using 75 features ---
              precision    recall  f1-score   support

         0.0     0.7937    0.9592    0.8686     11517
         1.0     0.5652    0.1754    0.2677      3483

    accuracy                         0.7772     15000
   macro avg     0.6794    0.5673    0.5682     15000
weighted avg     0.7406    0.7772    0.7291     15000

AUC-ROC: 0.7195


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

none really did well when it came to predicting the 1.0 values, though 0.0 is good. likely due to skewed target values. Let's try to balance that a bit.


### SMOTE

In [60]:
#introducing SMOTE
!pip install --upgrade pip
!pip install imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [61]:


X = traindf[best_of_both]
y = traindf['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

models = {
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    print(f"\n--- Evaluation for {name} ---")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(classification_report(y_test, y_pred, digits=4))

    # Try to get predicted probabilities or decision function
    try:
        y_proba = pipeline.predict_proba(X_test)[:, 1]
    except AttributeError:
        try:
            y_proba = pipeline.decision_function(X_test)
        except AttributeError:
            y_proba = None

    if y_proba is not None:
        auc = roc_auc_score(y_test, y_proba)
        print(f"AUC-ROC: {auc:.4f}")
    else:
        print("AUC-ROC could not be computed (model lacks probability or decision output).")





--- Evaluation for RandomForest ---
              precision    recall  f1-score   support

         0.0     0.8021    0.8917    0.8445     11496
         1.0     0.4392    0.2783    0.3407      3504

    accuracy                         0.7484     15000
   macro avg     0.6207    0.5850    0.5926     15000
weighted avg     0.7173    0.7484    0.7268     15000

AUC-ROC: 0.6708

--- Evaluation for LogisticRegression ---
              precision    recall  f1-score   support

         0.0     0.8431    0.7358    0.7858     11496
         1.0     0.3886    0.5508    0.4557      3504

    accuracy                         0.6926     15000
   macro avg     0.6158    0.6433    0.6207     15000
weighted avg     0.7369    0.6926    0.7087     15000

AUC-ROC: 0.6985

--- Evaluation for XGBoost ---
              precision    recall  f1-score   support

         0.0     0.7925    0.9384    0.8593     11496
         1.0     0.4895    0.1938    0.2777      3504

    accuracy                         0

In [62]:


X = traindf[top_target_corr_cols]
y = traindf['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

models = {
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    print(f"\n--- Evaluation for {name} ---")
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    print(classification_report(y_test, y_pred, digits=4))

    # Try to get predicted probabilities or decision function
    try:
        y_proba = pipeline.predict_proba(X_test)[:, 1]
    except AttributeError:
        try:
            y_proba = pipeline.decision_function(X_test)
        except AttributeError:
            y_proba = None

    if y_proba is not None:
        auc = roc_auc_score(y_test, y_proba)
        print(f"AUC-ROC: {auc:.4f}")
    else:
        print("AUC-ROC could not be computed (model lacks probability or decision output).")





--- Evaluation for RandomForest ---
              precision    recall  f1-score   support

         0.0     0.8126    0.8986    0.8534     11496
         1.0     0.4902    0.3199    0.3872      3504

    accuracy                         0.7634     15000
   macro avg     0.6514    0.6092    0.6203     15000
weighted avg     0.7372    0.7634    0.7445     15000

AUC-ROC: 0.7149

--- Evaluation for LogisticRegression ---
              precision    recall  f1-score   support

         0.0     0.8585    0.7202    0.7833     11496
         1.0     0.3994    0.6104    0.4828      3504

    accuracy                         0.6945     15000
   macro avg     0.6289    0.6653    0.6330     15000
weighted avg     0.7512    0.6945    0.7131     15000

AUC-ROC: 0.7150

--- Evaluation for XGBoost ---
              precision    recall  f1-score   support

         0.0     0.8066    0.9265    0.8624     11496
         1.0     0.5292    0.2711    0.3586      3504

    accuracy                         0

One last attempt. Let's try to explore the parameters. Random forest did best, so we'll stick with it and the cor list.

In [63]:
from sklearn.model_selection import GridSearchCV

X = traindf[top_target_corr_cols]
y = traindf['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# Grid of hyperparameters to try
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
    'model__penalty': ['l2'],            # 'l1' works only with 'liblinear' solver so sticking with this
    'model__solver': ['lbfgs', 'liblinear']
}

# Set up Grid Search
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring='roc_auc',     # optimize for AUC
    cv=5,                  # 5-fold cross-validation
    n_jobs=-1,             # Use all cores
    verbose=1
)

grid_search.fit(X_train, y_train)

# Best model from grid
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n--- Grid Search Evaluation for LogisticRegression ---")
print("Best Params:", grid_search.best_params_)
print(classification_report(y_test, y_pred, digits=4))

# AUC
y_proba = best_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.4f}")


Fitting 5 folds for each of 10 candidates, totalling 50 fits

--- Grid Search Evaluation for LogisticRegression ---
Best Params: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'liblinear'}
              precision    recall  f1-score   support

         0.0     0.7941    0.9590    0.8688     11517
         1.0     0.5674    0.1777    0.2707      3483

    accuracy                         0.7776     15000
   macro avg     0.6807    0.5684    0.5697     15000
weighted avg     0.7414    0.7776    0.7299     15000

AUC-ROC: 0.7194
